# DCT Basis Benchmarks for Neural Image Compression: Revealing Frequency Dependent Biases

### Libraries

In [1]:
import warnings
from pathlib import Path

import torch

from loaders import load_model, get_available_models
from functions import (
    evaluate_frequency_response,
    build_summary_row,
    merge_and_save_csv,
    save_all_artifacts,
)

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='tqdm')

device = torch.device("cuda")

### Experiment configuration

In [ ]:
model_range = get_available_models(include_codecs=True)
quality_range = [1, 2, 3, 4, 5, 6]
size_range = [64, 128, 256, 512, 1024]
# TCM-specific settings
tcm_size_range = [256, 512, 1024]
tcm_p_range = [64, 128]
print('Models to evaluate:', model_range)

Models to evaluate: ['tcm', 'cheng2020-anchor', 'cheng2020-attn', 'bmshj2018-factorized', 'bmshj2018-hyperprior', 'mbt2018-mean', 'mbt2018', 'webp', 'jpegxl', 'jpeg']


In [ ]:
# Evaluation parameters
NUM_RUNS = 100 # average over 100 runs
SEED = 42
SAVE_CE_WINDOW = 2

# Where to store results
BASE_RESULTS_DIR = Path('results')
BASE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BASE_DIR = "/home/nkalmykov/compressai_project/experiments"

In [ ]:
all_runs = []
for model_name in model_range:
    sizes_iter = tcm_size_range if model_name == 'tcm' else size_range
    qp_iter = tcm_p_range if model_name == 'tcm' else quality_range

    for size in sizes_iter:
        print(f"\n=== Model: {model_name} | Size: {size}x{size} ===")
        size_dir = BASE_RESULTS_DIR / model_name / str(size)
        per_size_rows = []

        for qp in qp_iter:
            is_tcm = model_name == 'tcm'
            p_val = qp if is_tcm else None
            q_val = None if is_tcm else qp

            print(f"Evaluating {'p=' + str(p_val) if is_tcm else 'quality ' + str(q_val)}...")
            q_dir = size_dir / (f"p_{p_val}" if is_tcm else f"q_{q_val}")

            # Load model and evaluate
            model = load_model(model_name, q_val, device, p=p_val or 128, base_dir=BASE_DIR)
            x_dct_rgb, x_hat, metrics = evaluate_frequency_response(
                model, size=size, device=device,
                show_plots=False, num_runs=NUM_RUNS, show_metric_plots=False, seed=SEED,
            )

            # Save artifacts (images + plots)
            save_all_artifacts(q_dir, x_dct_rgb, x_hat, metrics)

            # Build summary row
            row = build_summary_row(
                model_name, size, metrics,
                quality=q_val, p=p_val, ce_window=SAVE_CE_WINDOW,
            )
            per_size_rows.append(row)
            all_runs.append(row)

        # Save per-size CSV
        df_size = merge_and_save_csv(per_size_rows, size_dir / 'metrics_summary.csv')
        print(df_size)

        # Update global CSV
        merge_and_save_csv(all_runs, BASE_RESULTS_DIR / 'all_metrics_summary.csv')

print("\nDone! Results saved to:", BASE_RESULTS_DIR)


=== Model: tcm | Size: 256x256 ===
Evaluating p=64...


/home/nkalmykov/tmla/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Summary (median over k; H in bits; CE window=2; runs=100): L_k=0.0280, ODR_k=0.0144, |Δc_k|=0.0044, s_k=0.0610, H_k=0.3652, CE_k(w=2)=0.9759
Evaluating p=128...
Summary (median over k; H in bits; CE window=2; runs=100): L_k=0.0019, ODR_k=0.0010, |Δc_k|=0.0003, s_k=0.0215, H_k=0.0337, CE_k(w=2)=0.9981
  Model     Size    p     L_k   L_low  L_high   ODR_k  |Delta_c_k|     s_k  \
0   tcm  256x256   64  0.0280  0.0141  0.0638  0.0144       0.0044  0.0610   
1   tcm  256x256  128  0.0019  0.0014  0.0054  0.0010       0.0003  0.0215   

   H_k_bits  CE_k(w=2)  
0    0.0659     0.9759  
1    0.0061     0.9981  

=== Model: tcm | Size: 512x512 ===
Evaluating p=64...
Summary (median over k; H in bits; CE window=2; runs=100): L_k=0.0267, ODR_k=0.0137, |Δc_k|=0.0033, s_k=0.0568, H_k=0.3694, CE_k(w=2)=0.9776
Evaluating p=128...
Summary (median over k; H in bits; CE window=2; runs=100): L_k=0.0019, ODR_k=0.0009, |Δc_k|=0.0004, s_k=0.0210, H_k=0.0341, CE_k(w=2)=0.9982
  Model     Size    p     L_k  

100%|██████████| 11.5M/11.5M [00:04<00:00, 2.95MB/s]


Summary (median over k; H in bits; CE window=2; runs=100): L_k=0.9007, ODR_k=0.9998, |Δc_k|=0.0501, s_k=0.1285, H_k=2.8794, CE_k(w=2)=0.4697
Evaluating quality 2...
Downloading: "https://compressai.s3.amazonaws.com/models/v1/bmshj2018-factorized-prior-2-87279a02.pth.tar" to /home/nkalmykov/.cache/torch/hub/checkpoints/bmshj2018-factorized-prior-2-87279a02.pth.tar


...

Summary (median over k; H in bits; CE window=2; runs=100): L_k=0.3382, ODR_k=0.2501, |Δc_k|=0.0127, s_k=0.1152, H_k=1.9100, CE_k(w=2)=0.9039
Evaluating quality 5...
Downloading: "https://compressai.s3.amazonaws.com/models/v1/mbt2018-5-b4a046dd.pth.tar" to /home/nkalmykov/.cache/torch/hub/checkpoints/mbt2018-5-b4a046dd.pth.tar


100%|██████████| 118M/118M [00:28<00:00, 4.29MB/s] 


Summary (median over k; H in bits; CE window=2; runs=100): L_k=0.1326, ODR_k=0.0763, |Δc_k|=0.0081, s_k=0.0905, H_k=1.0787, CE_k(w=2)=0.9415
Evaluating quality 6...
Downloading: "https://compressai.s3.amazonaws.com/models/v1/mbt2018-6-7052e5ea.pth.tar" to /home/nkalmykov/.cache/torch/hub/checkpoints/mbt2018-6-7052e5ea.pth.tar


100%|██████████| 118M/118M [00:33<00:00, 3.73MB/s] 


Summary (median over k; H in bits; CE window=2; runs=100): L_k=0.1344, ODR_k=0.0775, |Δc_k|=0.0052, s_k=0.0740, H_k=1.0284, CE_k(w=2)=0.9501
     Model   Size  q     L_k   L_low  L_high   ODR_k  |Delta_c_k|     s_k  \
0  mbt2018  64x64  1  0.7464  0.4477  0.9599  0.8994       0.0254  0.1485   
1  mbt2018  64x64  2  0.4967  0.3174  0.8174  0.4570       0.0120  0.1295   
2  mbt2018  64x64  3  0.3836  0.2254  0.8003  0.3015       0.0183  0.1155   
3  mbt2018  64x64  4  0.3382  0.1500  0.6961  0.2501       0.0127  0.1152   
4  mbt2018  64x64  5  0.1326  0.0763  0.4798  0.0763       0.0081  0.0905   
5  mbt2018  64x64  6  0.1344  0.0747  0.2754  0.0775       0.0052  0.0740   

   H_k_bits  CE_k(w=2)  
0    0.7364     0.8381  
1    0.6043     0.8231  
2    0.5374     0.8420  
3    0.4593     0.9039  
4    0.2594     0.9415  
5    0.2473     0.9501  

=== Model: mbt2018 | Size: 128x128 ===
Evaluating quality 1...
Summary (median over k; H in bits; CE window=2; runs=100): L_k=0.7016, ODR_k=0.8